# NaukriAI — CV Parser Fine-Tuning (Google Colab)

**Runtime required:** GPU (T4 or better)  
In Colab: **Runtime → Change runtime type → T4 GPU**

This notebook fine-tunes Mistral 7B using QLoRA on our 101-example CV parsing dataset.  
Output: a LoRA adapter + GGUF model you can load into Ollama.

In [ ]:
import torch, gc, os

# Clear any leftover GPU state from previous runs
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free = torch.cuda.mem_get_info()[0] / 1e9
    total = props.total_memory / 1e9
    print(f'GPU:   {props.name}')
    print(f'VRAM:  {total:.1f} GB total | {free:.1f} GB free')
    if free < 10:
        print('\n⚠️  Less than 10 GB free — do Runtime → Disconnect and Delete Runtime, then reconnect.')
    else:
        print('✓ Enough VRAM to proceed.')
else:
    print('❌ No GPU — Runtime → Change runtime type → T4 GPU')

In [ ]:
# Install dependencies (takes ~3 minutes)
!pip install -q unsloth trl peft bitsandbytes datasets transformers accelerate

In [ ]:
import os
from google.colab import files

print('Upload dataset_examples.jsonl from your PC...')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded. Please select dataset_examples.jsonl and try again.')

# Write to a fixed path regardless of uploaded filename
raw = list(uploaded.values())[0]
DATASET_PATH = '/content/dataset_examples.jsonl'
with open(DATASET_PATH, 'wb') as f:
    f.write(raw)

lines = sum(1 for l in open(DATASET_PATH) if l.strip())
print(f'✓ Saved to {DATASET_PATH} ({lines} examples, {len(raw)/1024:.1f} KB)')

In [ ]:
import json
from datasets import Dataset

# DATASET_PATH set by Cell 3 — fail early if not set
if not os.path.exists(DATASET_PATH):
    raise RuntimeError(f'Dataset not found at {DATASET_PATH}. Re-run Cell 3 and upload the file.')

OUTPUT_DIR = '/content/naukri-cv-parser-lora'
MODEL_NAME  = 'unsloth/Phi-3-mini-4k-instruct-bnb-4bit'

examples = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            ex     = json.loads(line)
            instr  = ex['instruction']
            inp    = ex.get('input', '')
            output = ex['output']
            if inp:
                text = f'<|user|>\n{instr}\n\n{inp}<|end|>\n<|assistant|>\n{output}<|end|>'
            else:
                text = f'<|user|>\n{instr}<|end|>\n<|assistant|>\n{output}<|end|>'
            examples.append({'text': text})

dataset = Dataset.from_list(examples)
print(f'✓ {len(dataset)} training examples loaded')
print(f'  Model: {MODEL_NAME}')
print(f'  Sample ({len(dataset[0]["text"])} chars):', dataset[0]['text'][:200])

In [ ]:
from unsloth import FastLanguageModel
import torch, gc

gc.collect()
torch.cuda.empty_cache()

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MAX_SEQ = 2048
print(f'VRAM: {vram_gb:.1f} GB | max_seq_length: {MAX_SEQ}')
print(f'Loading {MODEL_NAME}...')

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)

free_after_load = torch.cuda.mem_get_info()[0] / 1e9
print(f'Free VRAM after model load: {free_after_load:.1f} GB')

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_up_proj', 'down_proj'],  # Phi-3 layer names
    lora_alpha=16,
    lora_dropout=0.05,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bf16_supported
import gc, torch

# Free any cached memory before training
gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=3,
        max_steps=300,
        learning_rate=2e-4,
        warmup_steps=20,
        lr_scheduler_type='cosine',
        fp16=not is_bf16_supported(),   # T4 → fp16=True, bf16=False
        bf16=is_bf16_supported(),       # A100/H100 → bf16=True, fp16=False
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        optim='adamw_8bit',
        weight_decay=0.01,
        dataloader_pin_memory=False,
        report_to='none',
        seed=42,
    ),
)

free_gb = torch.cuda.mem_get_info()[0] / 1e9
print(f'Free VRAM before training: {free_gb:.1f} GB')
print('Starting QLoRA fine-tuning (~40-50 min on T4)...')
trainer.train()

In [ ]:
# Save LoRA adapter
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'LoRA adapter saved to {OUTPUT_DIR}')

In [ ]:
# Export to GGUF for Ollama (q4_k_m = best quality/size tradeoff)
gguf_path = OUTPUT_DIR + '/model.gguf'
print('Exporting to GGUF format (takes 5-10 min)...')
model.save_pretrained_gguf(OUTPUT_DIR, tokenizer, quantization_method='q4_k_m')
print(f'GGUF saved: {gguf_path}')

In [ ]:
# Download the GGUF file to your PC
from google.colab import files
import os

# Zip the output directory for download
!zip -r /content/naukri-cv-parser-lora.zip {OUTPUT_DIR}
files.download('/content/naukri-cv-parser-lora.zip')

print('\n=== DONE ===')
print('After downloading:')
print('1. Extract to ai-engine/models/naukri-cv-parser-lora/')
print('2. Copy the .gguf file to ai-engine/setup/')
print('3. Run: ollama create naukri-cv-parser -f ai-engine/setup/Modelfile')
print('4. Set OLLAMA_MODEL=naukri-cv-parser in your .env')

## After Training — Integrate with Ollama

1. Extract downloaded zip → `ai-engine/models/naukri-cv-parser-lora/`
2. Update `ai-engine/setup/Modelfile`:
   ```
   FROM ./models/naukri-cv-parser-lora/model.gguf
   PARAMETER temperature 0.1
   PARAMETER num_predict 4096
   ```
3. Create model in Ollama: `ollama create naukri-cv-parser -f ai-engine/setup/Modelfile`
4. Set env: `OLLAMA_MODEL=naukri-cv-parser` in your `.env`
5. Restart the AI engine